In [0]:
# =============================================================================
# NOTEBOOK  : bronze_store_inventory
# PURPOSE   : Ingest store_inventory_snapshots.jsonl → bronze.store_inventory
# LAYER     : Bronze (raw ingestion — no transformation)
# FREQUENCY : Every 15 minutes (Autoloader availableNow)
# FORMAT    : JSONL (one JSON object per line)
# NOTE      : temperature_reading is a nested JSON string (when present):
#             {"sensor_id":"SENSOR_27","temperature_celsius":1.62,"humidity":61.64}
#             null for non temperature-sensitive products
#             Kept as raw string in bronze — flattened in silver
#             expiry_date null for non-perishables — kept as-is in bronze
# =============================================================================

# ── Imports & Config ──────────────────────────────────────────────────
import sys, json
sys.path.insert(0, "/Workspace/Shared/mini_project_a3t7")

from utils.audit import start_audit, end_audit
from utils.quality_checks import assert_not_empty
from utils.schema_registry import BRONZE_STORE_INVENTORY_SCHEMA

from pyspark.sql.functions import current_timestamp, lit, col, to_date

with open("/Workspace/Shared/mini_project_a3t7/config/config.json") as f:
    cfg = json.load(f)

SOURCE_PATH  = cfg["landing_paths"]["store_inventory"]
TARGET_TABLE = cfg["tables"]["bronze_store_inventory"]
CHECKPOINT   = cfg["checkpoint_paths"]["bronze_store_inventory"]
PIPELINE     = "bronze_store_inventory"

In [0]:
# ── Audit + Read + Write ─────────────────────────────────────────────
run_id = start_audit(spark, PIPELINE, TARGET_TABLE)

try:
    # ── READ (Autoloader — availableNow) ─────────────────────────────────────
    # JSONL = one JSON object per line
    # Autoloader reads each line as a separate row
    # temperature_reading is a nested JSON string inside the JSONL record —
    # Spark reads it as a plain STRING since schema declares StringType
    # multiLine must be false (default) for JSONL — each line is one record
    df = (
        spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "json")
        .option("multiLine", "false")
        .option("cloudFiles.schemaLocation", CHECKPOINT + "/schema")
        .schema(BRONZE_STORE_INVENTORY_SCHEMA)
        .load(SOURCE_PATH)
    )

    # ── ADD AUDIT COLUMNS ─────────────────────────────────────────────────────
    df = (
        df
        .withColumn("source_file",     col("_metadata.file_path"))
        .withColumn("ingested_at",     current_timestamp())
        .withColumn("ingested_date",   to_date(current_timestamp()).cast("string"))
        .withColumn("pipeline_run_id", lit(run_id))
    )

    # ── WRITE ─────────────────────────────────────────────────────────────────
    query = (
        df.writeStream
        .format("delta")
        .outputMode("append")
        .option("checkpointLocation", CHECKPOINT)
        .partitionBy("ingested_date")
        .trigger(availableNow=True)
        .toTable(TARGET_TABLE)
    )

    query.awaitTermination()

    # ── QUALITY CHECK ─────────────────────────────────────────────────────────
    written_df = (
        spark.read.table(TARGET_TABLE)
        .where(f"pipeline_run_id = '{run_id}'")
    )
    rows_written = written_df.count()
    assert_not_empty(written_df, PIPELINE)

    print(f"[WRITE] {rows_written} rows written to {TARGET_TABLE}")

    # ── END AUDIT (SUCCESS) ───────────────────────────────────────────────────
    end_audit(spark, run_id, "SUCCESS", rows_written=rows_written)

except Exception as e:
    end_audit(spark, run_id, "FAILED", error=str(e))
    raise